<a href="https://colab.research.google.com/github/mynameisnitya/AI-Chatbot/blob/main/01_MIMIC_Exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mount Drive for Reading dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
# @title
import zipfile

zip_path = "/content/drive/MyDrive/MIMIC-III/mimic-iii-clinical-database-1.4.zip"
extract_path = "/content/drive/MyDrive/MIMIC-III/data"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

#LOAD CORE TABLES


In [90]:

import pandas as pd
import numpy as np

BASE_PATH = "/content/drive/MyDrive/MIMIC-III/data/mimic-iii-clinical-database-1.4/"

# Small tables (safe)

patients = pd.read_csv(BASE_PATH + "PATIENTS.csv.gz")
admissions = pd.read_csv(BASE_PATH + "ADMISSIONS.csv.gz")
diagnoses = pd.read_csv(BASE_PATH + "DIAGNOSES_ICD.csv.gz")
d_icd = pd.read_csv(BASE_PATH + "D_ICD_DIAGNOSES.csv.gz")

print("PATIENTS:", patients.shape)
print("ADMISSIONS:", admissions.shape)
print("DIAGNOSES:", diagnoses.shape)
print("D_ICD:", d_icd.shape)

print("Loaded data")


PATIENTS: (46520, 8)
ADMISSIONS: (58976, 19)
DIAGNOSES: (651047, 5)
D_ICD: (14567, 4)
Loaded data


In [93]:
patients.head()


,ROW_ID,SUBJECT_ID,GENDER,DOB,DOD,DOD_HOSP,DOD_SSN,EXPIRE_FLAG
0,234,249,F,2075-03-13 00:00:00,NaN,NaN,NaN,0
1,235,250,F,2164-12-27 00:00:00,2188-11-22 00:00:00,2188-11-22 00:00:00,NaN,1
2,236,251,M,2090-03-15 00:00:00,NaN,NaN,NaN,0
3,237,252,M,2078-03-06 00:00:00,NaN,NaN,NaN,0
4,238,253,F,2089-11-26 00:00:00,NaN,NaN,NaN,0


In [94]:
admissions.head()


,ROW_ID,SUBJECT_ID,HADM_ID,ADMITTIME,DISCHTIME,DEATHTIME,ADMISSION_TYPE,ADMISSION_LOCATION,DISCHARGE_LOCATION,INSURANCE,LANGUAGE,RELIGION,MARITAL_STATUS,ETHNICITY,EDREGTIME,EDOUTTIME,DIAGNOSIS,HOSPITAL_EXPIRE_FLAG,HAS_CHARTEVENTS_DATA
0,21,22,165315,2196-04-09 12:26:00,2196-04-10 15:54:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,DISC-TRAN CANCER/CHLDRN H,Private,NaN,UNOBTAINABLE,MARRIED,WHITE,2196-04-09 10:06:00,2196-04-09 13:24:00,BENZODIAZEPINE OVERDOSE,0,1
1,22,23,152223,2153-09-03 07:15:00,2153-09-08 19:10:00,NaN,ELECTIVE,PHYS REFERRAL/NORMAL DELI,HOME HEALTH CARE,Medicare,NaN,CATHOLIC,MARRIED,WHITE,NaN,NaN,CORONARY ARTERY DISEASE\CORONARY ARTERY BYPASS...,0,1
2,23,23,124321,2157-10-18 19:34:00,2157-10-25 14:00:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME HEALTH CARE,Medicare,ENGL,CATHOLIC,MARRIED,WHITE,NaN,NaN,BRAIN MASS,0,1
3,24,24,161859,2139-06-06 16:14:00,2139-06-09 12:48:00,NaN,EMERGENCY,TRANSFER FROM HOSP/EXTRAM,HOME,Private,NaN,PROTESTANT QUAKER,SINGLE,WHITE,NaN,NaN,INTERIOR MYOCARDIAL INFARCTION,0,1
4,25,25,129635,2160-11-02 02:06:00,2160-11-05 14:55:00,NaN,EMERGENCY,EMERGENCY ROOM ADMIT,HOME,Private,NaN,UNOBTAINABLE,MARRIED,WHITE,2160-11-02 01:01:00,2160-11-02 04:27:00,ACUTE CORONARY SYNDROME,0,1


In [95]:
diagnoses.head()

,ROW_ID,SUBJECT_ID,HADM_ID,SEQ_NUM,ICD9_CODE
0,1297,109,172335,1.0,40301
1,1298,109,172335,2.0,486
2,1299,109,172335,3.0,58281
3,1300,109,172335,4.0,5855
4,1301,109,172335,5.0,4254


## Check for Missing Values

In [100]:
print("----NULL VALUES---")
print("Patients: \n",patients.isnull().sum())
print("---------------------------------")
print("Admissions: \n",admissions.isnull().sum())
print("---------------------------------")
print("Diagnoses: \n",diagnoses.isnull().sum())

----NULL VALUES---
Patients: 
 ROW_ID             0
SUBJECT_ID         0
GENDER             0
DOB                0
DOD            30761
DOD_HOSP       36546
DOD_SSN        33142
EXPIRE_FLAG        0
dtype: int64
---------------------------------
Admissions: 
 ROW_ID                      0
SUBJECT_ID                  0
HADM_ID                     0
ADMITTIME                   0
DISCHTIME                   0
DEATHTIME               53122
ADMISSION_TYPE              0
ADMISSION_LOCATION          0
DISCHARGE_LOCATION          0
INSURANCE                   0
LANGUAGE                25332
RELIGION                  458
MARITAL_STATUS          10128
ETHNICITY                   0
EDREGTIME               28099
EDOUTTIME               28099
DIAGNOSIS                  25
HOSPITAL_EXPIRE_FLAG        0
HAS_CHARTEVENTS_DATA        0
dtype: int64
---------------------------------
Diagnoses: 
 ROW_ID         0
SUBJECT_ID     0
HADM_ID        0
SEQ_NUM       47
ICD9_CODE     47
dtype: int64


# Statistics

In [101]:
print("Unique Patients:",
      patients["SUBJECT_ID"].nunique())

print("Unique Admissions:",
      admissions["HADM_ID"].nunique())

Unique Patients: 46520
Unique Admissions: 58976


In [102]:
diabetes_codes = diagnoses[
    diagnoses["ICD9_CODE"].astype(str).str.startswith("250")
]

print(diabetes_codes.shape)

(16454, 5)


## Diabetes patients

In [103]:
diabetes_patients = diabetes_codes[
    "SUBJECT_ID"
].nunique()

print("Diabetes Patients:",
      diabetes_patients)

Diabetes Patients: 10318


## CVD Patients

In [104]:
diagnoses["ICD9_CODE"] = diagnoses[
    "ICD9_CODE"
].astype(str)

cvd = diagnoses[
    diagnoses["ICD9_CODE"].str.match(
        r'^(39|40|41|42|43|44|45)'
    )
]

print(
    "CVD Patients:",
    cvd["SUBJECT_ID"].nunique()
)

CVD Patients: 32503


# Load Lab Events

In [105]:
labevents = pd.read_csv(
    BASE_PATH + "LABEVENTS.csv.gz",
    usecols=["SUBJECT_ID","ITEMID","VALUENUM"]
)

print(labevents.shape)

(27854055, 3)


## Load Lab Dictionary

In [106]:
d_labitems = pd.read_csv(
    BASE_PATH + "D_LABITEMS.csv.gz"
)

print(d_labitems.shape)

(753, 6)


## Glucose search

In [107]:
d_labitems[
    d_labitems["LABEL"].str.contains(
        "glucose",
        case=False,
        na=False
    )
][["ITEMID","LABEL"]].head(20)

,ITEMID,LABEL
136,50809,Glucose
169,50842,"Glucose, Ascites"
258,50931,Glucose
340,51014,"Glucose, CSF"
348,51022,"Glucose, Joint Fluid"
360,51034,"Glucose, Body Fluid"
379,51053,"Glucose, Pleural"
410,51084,"Glucose, Urine"
677,51478,Glucose
728,51529,Estimated Actual Glucose


 ## Cholesterol Search

In [108]:
d_labitems[
    d_labitems["LABEL"].str.contains(
        "cholesterol",
        case=False,
        na=False
    )
][["ITEMID","LABEL"]]

,ITEMID,LABEL
126,51472,Cholesterol Crystals
167,50840,"Cholesterol, Ascites"
230,50903,Cholesterol Ratio (Total/HDL)
231,50904,"Cholesterol, HDL"
232,50905,"Cholesterol, LDL, Calculated"
233,50906,"Cholesterol, LDL, Measured"
234,50907,"Cholesterol, Total"
357,51031,"Cholesterol, Body Fluid"
377,51051,"Cholesterol, Pleural"


## Heart Reate Search

In [110]:
d_items = pd.read_csv(
    BASE_PATH + "D_ITEMS.csv.gz"
)

print(d_items.shape)
d_items[
    d_items["LABEL"].str.contains(
        "heart rate",
        case=False,
        na=False
    )
][["ITEMID","LABEL"]].head(20)

(12487, 10)


,ITEMID,LABEL
475,211,Heart Rate
1897,3494,Lowest Heart Rate
11498,220045,Heart Rate
11499,220046,Heart rate Alarm - High
11500,220047,Heart Rate Alarm - Low


## Search Blood pressure

In [112]:
d_items[
    d_items["LABEL"].str.contains(
        "systolic",
        case=False,
        na=False
    )
][["ITEMID","LABEL"]].head(20)


,ITEMID,LABEL
295,6,ABP [Systolic]
320,51,Arterial BP [Systolic]
671,442,Manual BP [Systolic]
682,455,NBP [Systolic]
705,480,Orthostat BP sitting [Systolic]
707,482,OrthostatBP standing [Systolic]
709,484,Orthostatic BP lying [Systolic]
715,492,PAP [Systolic]
1437,666,Systolic Unloading
1748,3313,BP Cuff [Systolic]


In [113]:
d_items[
    d_items["LABEL"].str.contains(
        "diastolic",
        case=False,
        na=False
    )
][["ITEMID","LABEL"]].head(20)

,ITEMID,LABEL
417,153,Diastolic Unloading
4633,8364,ABP [Diastolic]
4637,8368,Arterial BP [Diastolic]
4709,8440,Manual BP [Diastolic]
4710,8441,NBP [Diastolic]
4713,8444,Orthostat BP sitting [Diastolic]
4714,8445,OrthostatBP standing [Diastolic]
4715,8446,Orthostatic BP lying [Diastolic]
4717,8448,PAP [Diastolic]
4759,8502,BP Cuff [Diastolic]


## Feature Extration

In [114]:
diagnoses["ICD9_CODE"] = diagnoses["ICD9_CODE"].astype(str)

diabetes_patients = diagnoses[
    diagnoses["ICD9_CODE"].str.startswith("250")
]

labels = (
    diabetes_patients.groupby("SUBJECT_ID")
    .size()
    .reset_index(name="DIABETES")
)

labels["DIABETES"] = 1

print(labels.shape)
labels.head()

(10318, 2)


,SUBJECT_ID,DIABETES
0,13,1
1,18,1
2,20,1
3,21,1
4,24,1


# Age and Gender

In [115]:
patients["DOB"] = pd.to_datetime(
    patients["DOB"],
    errors="coerce"
)

admissions["ADMITTIME"] = pd.to_datetime(
    admissions["ADMITTIME"],
    errors="coerce"
)

demo = admissions.merge(
    patients,
    on="SUBJECT_ID"
)

demo["AGE"] = (
    demo["ADMITTIME"].dt.year
    - demo["DOB"].dt.year
)

demo.loc[demo["AGE"] > 89, "AGE"] = 90

demo = demo[
    ["SUBJECT_ID", "AGE", "GENDER"]
]

demo.head()

,SUBJECT_ID,AGE,GENDER
0,22,65,F
1,23,71,M
2,23,75,M
3,24,39,M
4,25,59,M


# Glucose Extraction

In [116]:
glucose = labevents[
    labevents["ITEMID"] == 50809
]

glucose = (
    glucose.groupby("SUBJECT_ID")
    ["VALUENUM"]
    .mean()
    .reset_index()
)

glucose.columns = [
    "SUBJECT_ID",
    "GLUCOSE"
]

print(glucose.shape)

(24058, 2)


## Cholestrolo Extraction

In [117]:
cholesterol = labevents[
    labevents["ITEMID"] == 50907
]

cholesterol = (
    cholesterol.groupby("SUBJECT_ID")
    ["VALUENUM"]
    .mean()
    .reset_index()
)

cholesterol.columns = [
    "SUBJECT_ID",
    "CHOLESTEROL"
]

print(cholesterol.shape)

(10480, 2)


In [118]:
print(labels.shape)
print(demo.shape)
print(glucose.shape)
print(cholesterol.shape)

(10318, 2)
(58976, 3)
(24058, 2)
(10480, 2)


## Extract Vitals

In [119]:
vital_ids = [220045, 220179, 220180]

chartevents = pd.read_csv(
    BASE_PATH + "CHARTEVENTS.csv.gz",
    usecols=["SUBJECT_ID", "ITEMID", "VALUENUM"],
    chunksize=1000000
)

vital_chunks = []

for chunk in chartevents:
    filtered = chunk[
        chunk["ITEMID"].isin(vital_ids)
    ]
    vital_chunks.append(filtered)

vitals = pd.concat(vital_chunks)

print(vitals.shape)

(5342598, 3)


## Heart Rate /BP Features Extraction

In [120]:
vitals = vitals.dropna(subset=["VALUENUM"])

vitals = (
    vitals.groupby(
        ["SUBJECT_ID", "ITEMID"]
    )["VALUENUM"]
    .mean()
    .unstack()
)

vitals = vitals.rename(
    columns={
        220045: "HEART_RATE",
        220179: "SYS_BP",
        220180: "DIA_BP"
    }
)

print(vitals.shape)
vitals.head()

(17717, 3)


ITEMID,HEART_RATE,SYS_BP,DIA_BP
SUBJECT_ID,,,
23,76.694444,107.062500,62.187500
34,55.864865,128.055556,59.888889
36,85.586538,127.950000,71.390000
85,106.276596,111.125000,63.687500
107,74.975904,110.776316,50.697368


## Merge

In [121]:
final_df = demo.copy()

final_df = final_df.merge(
    glucose,
    on="SUBJECT_ID",
    how="left"
)

final_df = final_df.merge(
    cholesterol,
    on="SUBJECT_ID",
    how="left"
)

final_df = final_df.merge(
    vitals,
    on="SUBJECT_ID",
    how="left"
)

final_df = final_df.merge(
    labels,
    on="SUBJECT_ID",
    how="left"
)

final_df["DIABETES"] = (
    final_df["DIABETES"]
    .fillna(0)
)

print(final_df.shape)

(58976, 9)


In [122]:
print(vitals.shape)

print(final_df.shape)

final_df.isnull().sum()

(17717, 3)
(58976, 9)


,0
SUBJECT_ID,0
AGE,0
GENDER,0
GLUCOSE,24768
CHOLESTEROL,42637
HEART_RATE,33824
SYS_BP,34050
DIA_BP,34050
DIABETES,0


In [123]:
ml_df = final_df[
    [
        "AGE",
        "GENDER",
        "GLUCOSE",
        "CHOLESTEROL",
        "HEART_RATE",
        "SYS_BP",
        "DIA_BP",
        "DIABETES"
    ]
]

print(ml_df.shape)

print(
    ml_df.dropna().shape
)

(58976, 8)
(5555, 8)
